## How to use AI Model

The model has already been trained and is running in a roboflow cloud instance. This script shows you how to get its predictions live on your camera.

In [2]:
import cv2
import json
import base64
import time
import traitlets
import ipywidgets.widgets

from IPython.display import display
from urllib import request, error
from jetbot import Robot, Camera, bgr8_to_jpeg
from jupyter_clickable_image_widget import ClickableImageWidget

API_KEY = "Ub1KVwtGHHdLLKRzoxdG"
API_URL = "https://serverless.roboflow.com/kais-workspace-stbmo/workflows/detect-count-and-visualize-3"

print("hello")

hello


## Test it with live feed

In [3]:
CONFIDENCE_THRESHOLD = 0.8

camera = Camera.instance(width=224, height=224) 
livefeed = ipywidgets.Image() 
camera_link = traitlets.dlink((camera, 'value'), (livefeed, 'value'), transform=bgr8_to_jpeg)
snapshot_widget = ipywidgets.Image(width=camera.width, height=camera.height)


def take_snapshot():
    frame = camera.value.copy()   # numpy array, BGR
    snapshot_widget.value = bgr8_to_jpeg(frame)
    return frame, snapshot_widget

def infer_frame(frame):
    ok, enc = cv2.imencode(".jpg", frame, [int(cv2.IMWRITE_JPEG_QUALITY), 90])
    if not ok:
        raise RuntimeError("Failed to JPEG-encode frame")
        
    
    image_b64 = base64.b64encode(enc.tobytes()).decode("utf-8")

    payload = {
        "api_key": API_KEY,
        "inputs": {
            "image": {
                "type": "base64",
                "value": image_b64
            },
            "confidence": CONFIDENCE_THRESHOLD
        }
    }

    data = json.dumps(payload).encode("utf-8")

    req = request.Request(
        url=API_URL,
        data=data,
        headers={
            "Content-Type": "application/json",
            "Accept": "application/json",
            "User-Agent": "python-urllib/3.6"
        },
        method="POST"
    )

    with request.urlopen(req, timeout=120) as resp:
        return json.loads(resp.read().decode("utf-8"))

frame, snapshot_widget = take_snapshot()
display(snapshot_widget)
result = infer_frame(frame)

outputs = result.get("outputs", [])
if outputs and "output_image" in outputs[0]:
    output_b64 = result["outputs"][0]["output_image"]["value"]
    output_widget = ipywidgets.Image(
        value=base64.b64decode(output_b64),
        format='jpg'
    )
    display(output_widget)
    predictions = result["outputs"][0]["predictions"]["predictions"]
    if predictions:
        for i, pred in enumerate(predictions):
            print("Prediction", i)
            print("  class      :", pred["class"])
            print("  confidence :", pred["confidence"])
            print("  x          :", pred["x"])
            print("  y          :", pred["y"])
            print("  width      :", pred["width"])
            print("  height     :", pred["height"])
            print("  class_id   :", pred["class_id"])
            print("  detection_id:", pred["detection_id"])
            print()
    else:
        print("No predictions found")
else:
    print("No output image found")

display(livefeed)

RuntimeError: Could not initialize camera.  Please see error trace.

Remember to stop a running camera or you may face initialization errors later. In case you do so, shut down kernels and restart.

In [3]:
camera.stop()

NameError: name 'camera' is not defined